In [2]:
!pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 61.7 MB/s eta 0:00:00


In [3]:
import os
import gurobipy as gp

# WLSライセンス情報を環境変数に設定（毎回必要）
os.environ['GRB_WLSACCESSID'] = '6620f8c0-8a08-4131-878a-afbbb5d18166'
os.environ['GRB_WLSSECRET'] = '05bc7798-6c06-40cb-99e9-74a422210724'
os.environ['GRB_LICENSEID'] = '2685889'

# Gurobiの環境を初期化（WLS指定）
env = gp.Env(empty=True)
env.setParam("WLSAccessID", os.environ["GRB_WLSACCESSID"])
env.setParam("WLSSecret", os.environ["GRB_WLSSECRET"])
env.setParam("LicenseID", int(os.environ["GRB_LICENSEID"]))
env.start()

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2685889
Academic license 2685889 - for non-commercial use only - registered to ev___@meiji.ac.jp


<gurobipy.Env, Parameter changes: WLSAccessID=(user-defined), WLSSecret=(user-defined), LicenseID=2685889>

In [7]:

import pandas as pd
from gurobipy import Model, GRB

# ExcelからλとRを読み込む
df = pd.read_excel("Data5.xlsx", sheet_name=0)

lambda_t = df['spot5'].tolist()[:48]
R_b = df['pri5'].dropna().tolist()[:8]
R2_b = df['sec5'].dropna().tolist()[:8]
S = df['Sdata'].dropna().tolist()[:10]

df_s = pd.read_excel("0704サイクル劣化.xlsx", sheet_name="s", header=None, usecols=range(4), nrows=4)
df_t = pd.read_excel("0704サイクル劣化.xlsx", sheet_name="t", header=None, usecols=range(4), nrows=4)


S_mat = df_s.values.tolist()  # 4×4リスト
T_mat = df_t.values.tolist()  # 4×4リスト


# 定数
alpha = 0.5
f = 0.95
p = 1.0
p2 = 1.0
delta_t = 0.5
B_const = 1000
D = 100 * B_const / 0.2
T = range(48)
T_q = range(49)
B = range(8)
I = range(10)
I2 = range(4)
J2 = range(4)

# モデル作成
model = Model("full_model",env=env)

# 変数定義
k0 = model.addVars(T, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="k0")
k1 = model.addVars(T, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="k1")
k2 = model.addVars(T, lb=0, ub=1, vtype=GRB.CONTINUOUS, name="k2")
Qr = model.addVars(T_q, vtype=GRB.CONTINUOUS, name="Qr")
Qr2 = model.addVars(T_q, vtype=GRB.CONTINUOUS, name="Qr2")
Qs = model.addVars(T_q, lb=0, vtype=GRB.CONTINUOUS, name="Qs")
P_buy = model.addVars(T, lb=0, vtype=GRB.CONTINUOUS, name="P_buy")
P_sel = model.addVars(T, lb=0, vtype=GRB.CONTINUOUS, name="P_sel")
a = model.addVars(T, vtype=GRB.BINARY, name="a")
delta_kW = model.addVars(B, lb=0, vtype=GRB.CONTINUOUS, name="delta_kW")
delta_kW2 = model.addVars(B, lb=0, vtype=GRB.CONTINUOUS, name="delta_kW2")
x = model.addVars(T, I, lb=0, vtype=GRB.CONTINUOUS, name="x")
z = model.addVars(T, I, vtype=GRB.BINARY, name="z")


# 新しい変数定義
U2 = model.addVars(range(4), T, vtype=GRB.BINARY, name="U2")  # U2[i, t]
W2 = model.addVars([1, 2], 4, T, lb=0, vtype=GRB.BINARY, name="W2")  # W2[l, i, t]
Z2 = model.addVars(4, 4, T, vtype=GRB.BINARY, name="Z2")  # Z2[l, i, t]
Z3 = model.addVars(4, 4, T, vtype=GRB.BINARY, name="Z3")  # Z3[l, i, t]

L2 = [1/3 for _ in range(4)]  # 1~3をインデックスとするなら L2[1], L2[2], L2[3] を使うように注意

x2 = model.addVars(4, 4, T, lb=0, vtype=GRB.CONTINUOUS, name="x2")
y2 = model.addVars(4, 4, T, lb=0, vtype=GRB.CONTINUOUS, name="y2")


A2 = model.addVars(4, 4, T, lb=0, vtype=GRB.BINARY, name="A2")

# L定義
L = [0.1 for _ in I]


# k0 + k1 + k2 = 1 #1
for t in T:
    model.addConstr(k0[t] + k1[t] + k2[t] == 1)

# 初期・終端制約 #2
model.addConstr(Qr[0] + Qs[0] + Qr2[0] == Qr[48] + Qs[48] + Qr2[48])

# Qr 再帰制約  #3
for t in T:
    b = t // 6 #3
    delta_kWh = p * delta_kW[b] * delta_t
    rhs = f * (1 - alpha) * delta_kWh - (alpha / (f * f)) * delta_kWh
    model.addConstr(Qr[t + 1] == Qr[t] + rhs)

# Qr2 再帰制約  #4
for t in T:
    b = t // 6
    delta_kWh2 = p2 * delta_kW2[b] * delta_t
    rhs2 = f * (1 - alpha) * delta_kWh2 - (alpha / (f * f)) * delta_kWh2
    model.addConstr(Qr2[t + 1] == Qr2[t] + rhs2)

# Qr, Qr2, Qs <= B制約 #5
for t in T_q:
    if t < 48:
        model.addConstr(Qr[t] <= B_const * k1[t])
        model.addConstr(Qr2[t] <= B_const * k2[t])
        model.addConstr(Qs[t] <= B_const * k0[t])
    else:
        model.addConstr(Qr[t] <= B_const)
        model.addConstr(Qr2[t] <= B_const)
        model.addConstr(Qs[t] <= B_const)

# P_buy, P_sel 制約 #6
for t in T:
    model.addConstr(P_buy[t] <= B_const * a[t])
    model.addConstr(P_sel[t] <= B_const * (1 - a[t]))

# Qs 再帰  #7
for t in T:
    rhs = f * P_buy[t] - P_sel[t] / (f * f)
    model.addConstr(Qs[t + 1] == Qs[t] + rhs)

# x-z関係の制約  #8
for t in T:
    for i in I:
        if i == 0:
            model.addConstr(x[t, i] >= L[i] * z[t, i])
            model.addConstr(x[t, i] <= L[i])
        elif i == 9:
            model.addConstr(x[t, i] >= 0)
            model.addConstr(x[t, i] <= L[i] * z[t, i - 1])
        else:
            model.addConstr(x[t, i] >= L[i] * z[t, i])
            model.addConstr(x[t, i] <= L[i] * z[t, i - 1])

# xの合計制約  #9
for t in T:
    b = t // 6
    rhs = (P_sel[t] + alpha * p * delta_kW[b] * delta_t + alpha * p2 * delta_kW2[b] * delta_t) / (B_const * f * f)
    model.addConstr(sum(x[t, i] for i in I) == rhs)

# Qr, Qr2 に ΔkWh に基づく上下限制約を追加（t=1〜47）#10
for t in T[1:]:
    b = t // 6
    delta_kWh = p * delta_kW[b] * delta_t
    lower_qr = (alpha / (f * f)) * delta_kWh
    upper_qr = B_const * k1[t] - f * (1 - alpha) * delta_kWh
    model.addConstr(Qr[t - 1] >= lower_qr)
    model.addConstr(Qr[t - 1] <= upper_qr)

    delta_kWh2 = p2 * delta_kW2[b] * delta_t
    lower_qr2 = (alpha / (f * f)) * delta_kWh2
    upper_qr2 = B_const * k2[t] - f * (1 - alpha) * delta_kWh2
    model.addConstr(Qr2[t - 1] >= lower_qr2)
    model.addConstr(Qr2[t - 1] <= upper_qr2)


# U2の初期値制約と合計制約を時間tごとに定義 #11
for t in T:
    #model.addConstr(U2[0, t] == 0, name=f"U2_0_is_0_t{t}")
    model.addConstr(
        sum(U2[i, t] for i in range(4)) == 1,
        name=f"U2_sum_to_1_t{t}"
    )


# i2, j2 = 4として明示
i2, j2 = 4, 4

for i in range(i2):

    for t in T:
        #ΣA2=1-Z
        model.addConstr(A2[1,1,t]+A2[1,2,t]+A2[1,3,t]==Z3[1,0,t])
        model.addConstr(A2[2,1,t]+A2[2,2,t]+A2[2,3,t]==Z3[2,0,t])
        model.addConstr(A2[3,1,t]+A2[3,2,t]+A2[3,3,t]==Z2[2,0,t])


        #ΣW2=Z2
        model.addConstr(W2[1,0,t]+W2[1,1,t]+W2[1,2,t]+W2[1,3,t]==Z2[1,0,t])
        model.addConstr(W2[1,0,t]+W2[1,1,t]+W2[1,2,t]+W2[1,3,t]==Z2[2,0,t])

        # W2[1][i][t] に関する論理積制約
        model.addConstr(0<=Z2[1,i,t]+U2[i,t]-2*W2[1,i,t])
        model.addConstr(1>=Z2[1,i,t]+U2[i,t]-2*W2[1,i,t])

        model.addConstr(0<=Z2[2,i,t]+U2[i,t]-2*W2[1,i,t])
        model.addConstr(1>=Z2[2,i,t]+U2[i,t]-2*W2[1,i,t])

        # x2の範囲制約
        model.addConstr(x2[1, i, t] >= L2[1] * W2[1, i, t], name=f"x2_1_{i}_{t}_geq")
        model.addConstr(x2[1, i, t] <= L2[1] * U2[i, t], name=f"x2_1_{i}_{t}_leq")

        model.addConstr(x2[2, i, t] >= L2[2] * W2[2, i, t], name=f"x2_2_{i}_{t}_geq")
        model.addConstr(x2[2, i, t] <= L2[2] * W2[1, i, t], name=f"x2_2_{i}_{t}_leq")

        model.addConstr(x2[3, i, t] >= 0, name=f"x2_3_{i}_{t}_geq")
        model.addConstr(x2[3, i, t] <= L2[3] * W2[2, i, t], name=f"x2_3_{i}_{t}_leq")




# x2[0][i][t] = U2[i][t]（i≠0）
for i in range(i2):
  if (i==0):
    for t in T:
      model.addConstr(x2[0,i,t]<=0)
      model.addConstr(x2[0,i,t]>=0)
  #else:
    #for t in T:
        #model.addConstr(x2[0, i, t] == U2[i, t], name=f"x2_0_{i}_{t}_eq_U2")

#x2&y2[1~3][0]==0
for i in range(1,4):
    for t in T:
      model.addConstr(x2[0,i,t]==0)
      model.addConstr(x2[i,0,t]==0)
      model.addConstr(y2[i,0,t]==0)
      model.addConstr(y2[0,i,t]==0)



# y2に関する制約（U2[i, t]を使うように変更）
for i in range(4):
  if (i==0):
    for t in T:
        model.addConstr(y2[i, 1, t] >= L2[1] * (U2[2, t] + U2[3, t]), name=f"y2_{i}_1_{t}_geq")
        model.addConstr(y2[i, 1, t] <= L2[1] * (U2[1, t] + U2[2, t] + U2[3, t]), name=f"y2_{i}_1_{t}_leq")

        model.addConstr(y2[i, 2, t] >= L2[2] * U2[3, t], name=f"y2_{i}_2_{t}_geq")
        model.addConstr(y2[i, 2, t] <= L2[2] * (U2[2, t] + U2[3, t]), name=f"y2_{i}_2_{t}_leq")

        model.addConstr(y2[i, 3, t] >= 0, name=f"y2_{i}_3_{t}_geq")
        model.addConstr(y2[i, 3, t] <= L2[3] * U2[3, t], name=f"y2_{i}_3_{t}_leq")

        if i == 0:
            model.addConstr(y2[i, 0, t] == 0, name=f"y2_{i}_0_{t}_eq_0")
        else:
            model.addConstr(y2[i, 0, t] == U2[0, t], name=f"y2_{i}_0_{t}_eq_U2_0")

  for t in T:
    model.addConstr(Z3[1,0,t]==1-Z2[1,0,t])


  if(i==1 or i==2):
      for t in T:
          model.addConstr(0<=Z3[i,0,t]+U2[1, t]-2*A2[i,1,t])
          model.addConstr(1>=Z3[i,0,t]+U2[1, t]-2*A2[i,1,t])
          model.addConstr(0<=Z3[i,0,t]+U2[2, t]-2*A2[i,2,t])
          model.addConstr(1>=Z3[i,0,t]+U2[2, t]-2*A2[i,2,t])
          model.addConstr(0<=Z3[i,0,t]+U2[3, t]-2*A2[i,3,t])
          model.addConstr(1>=Z3[i,0,t]+U2[3, t]-2*A2[i,3,t])

  if(i==1 or i==2 or i==3):
      for t in T:
          model.addConstr(L2[1]*(A2[i,2,t]+A2[i,3,t])<=y2[i,1,t])
          model.addConstr(L2[1]*(A2[i,1,t]+A2[i,2,t]+A2[i,3,t])>=y2[i,1,t])
          model.addConstr(L2[3]*(A2[i,3,t])<=y2[i,2,t])
          model.addConstr(L2[2]*(A2[i,2,t]+A2[1,3,t])>=y2[i,2,t])
          model.addConstr(0<=y2[i,3,t])
          model.addConstr(L2[3]*(A2[i,3,t])>=y2[i,3,t])

  if(i==2):
      for t in T:
          model.addConstr(0<=Z2[1,0,t]+(1-Z2[2,0,t])-2*(Z3[2,0,t]))
          model.addConstr(1>=Z2[1,0,t]+(1-Z2[2,0,t])-2*(Z3[2,0,t]))

  if(i==3):
      for t in T:
          model.addConstr(0<=Z2[i-1,0,t]+U2[1, t]-2*A2[i,1,t])
          model.addConstr(1>=Z2[i-1,0,t]+U2[1, t]-2*A2[i,1,t])
          model.addConstr(0<=Z2[i-1,0,t]+U2[2, t]-2*A2[i,2,t])
          model.addConstr(1>=Z2[i-1,0,t]+U2[2, t]-2*A2[i,2,t])
          model.addConstr(0<=Z2[i-1,0,t]+U2[3, t]-2*A2[i,3,t])
          model.addConstr(1>=Z2[i-1,0,t]+U2[3, t]-2*A2[i,3,t])











for t in T:
    b = t // 6  # 6刻みの区切り

    # ΔkWhおよびΔkWh2
    delta_kWh = p * delta_kW[b] * delta_t
    delta_kWh2 = p2 * delta_kW2[b] * delta_t

    # Σx2 = (Qs + Qr + Qr2) / B
    model.addConstr(
        sum(x2[i, j, t] for i in range(4) for j in range(4)) == (Qs[t] + Qr[t] + Qr2[t]) / B_const,
        name=f"sum_x2_constraint_t{t}"
    )

    # Σy2 = (P_sel + ΔkWh + ΔkWh2) / (B * f^2)
    model.addConstr(
        sum(y2[i, j, t] for i in range(4) for j in range(4)) == (P_sel[t] + alpha * delta_kWh + (1 - alpha) * delta_kWh2) / (B_const * f * f),
        name=f"sum_y2_constraint_t{t}"
    )









# 目的関数
expr = (
    sum(lambda_t[t] * (P_buy[t] - P_sel[t]) for t in T)
    - sum(6 * R_b[b] * delta_kW[b] for b in B)
    - sum(6 * R2_b[b] * delta_kW2[b] for b in B)
    + sum(
        (S_mat[i][j] * x2[i, j, t] + T_mat[i][j] * y2[i, j, t])/2
        for i in range(len(S_mat))
        for j in range(len(S_mat[0]))
        for t in T
    )
)


model.setObjective(expr, GRB.MINIMIZE)

# 最適化
model.optimize()

# 出力
if model.status == GRB.OPTIMAL:
    print("\n--- x2[i][j][t] の値 ---")
    for i in range(4):
        for j in range(4):
            for t in T:
                val = x2[i, j, t].X
                if abs(val) > 1e-6:  # 0以外の値だけ表示（ノイズ除去）
                    print(f"x2[{i}][{j}][{t}] = {val:.4f}")

    print("\n--- y2[i][j][t] の値 ---")
    for i in range(4):
        for j in range(4):
            for t in T:

                val = y2[i, j, t].X
                if abs(val) > 1e-6:
                    print(f"y2[{i}][{j}][{t}] = {val:.4f}")

if model.status == GRB.OPTIMAL:
    print("\n--- U2[i][t] の値を横方向に表示 ---")
    for t in T:
        line = ""
        for i in range(4):
            val = U2[i, t].X
            line += f"U2[{i},{t}]={val:.1f}, "
        print(line.rstrip(", "))


if model.status == GRB.OPTIMAL:
    print("\n--- Z2[i][t] の値を横方向に表示 ---")
    for t in T:
        line = ""
        for i in range(1,3):
          for j in range(4):
            val = Z2[i,j , t].X
            line += f"Z2[{i},{j},{t}]={val:.1f}, "
        print(line.rstrip(", "))


if model.status == GRB.OPTIMAL:
    print("\n--- W2[i][t] の値を横方向に表示 ---")
    for t in T:
        line = ""
        for i in range(1,3):
          for j in range(4):
            val = W2[i,j , t].X
            line += f"W2[{i},{j},{t}]={val:.1f}, "
        print(line.rstrip(", "))

if model.status == GRB.OPTIMAL:
    print("\n--- (Qs + Qr + Qr2) / B_const の値 ---")
    for t in T_q:
        val = (Qs[t].X + Qr[t].X + Qr2[t].X) / B_const
        print(f"t={t}: {(Qs[t].X):.1f} + {(Qr[t].X):.1f} + {(Qr2[t].X):.1f} = {val:.4f}")

if model.status == GRB.OPTIMAL:
    print("\n--- (P_sel + delta_kWh + delta_kWh2) / (f^2 * B_const) の値 ---")
    for t in T:
        b = t // 6
        delta_kWhd =alpha * p * delta_kW[b].X * delta_t
        delta_kWh2d =(1 - alpha) * p2 * delta_kW2[b].X * delta_t
        denom = f * f * B_const
        result = (P_sel[t].X + delta_kWhd + delta_kWh2d) / denom
        print(f"t={t}: P_sel={P_sel[t].X:.2f}, Δ1={delta_kWhd:.2f}, Δ2={delta_kWh2d:.2f} → {result:.4f}")


if model.status == GRB.OPTIMAL:
    print("\n--- A2[i][j][t] の値 ---")
    for t in T:
        line = ""
        for i in range(4):
            for j in range(4):
                val = A2[i, j, t].X
                if abs(val) > 1e-6:
                    line += f"A2[{i},{j},{t}]={val:.0f}, "
        if line:
            print(line.rstrip(", "))


else:
    print("最適解が得られませんでした。")







Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 22.04.4 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Academic license 2685889 - for non-commercial use only - registered to ev___@meiji.ac.jp
Optimize a model with 7680 rows, 5827 columns and 21551 nonzeros
Model fingerprint: 0x1073031d
Variable types: 2419 continuous, 3408 integer (3408 binary)
Coefficient statistics:
  Matrix range     [3e-04, 1e+03]
  Objective range  [4e-05, 8e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e-01, 1e+03]
Presolve removed 6517 rows and 4452 columns
Presolve time: 0.12s
Presolved: 1163 rows, 1375 columns, 4180 nonzeros
Variable types: 895 continuous, 480 integer (480 binary)
Found heuristic solution: objective 0.0000000

Root relaxation: objective -2.417805e+05, 671 iterations, 0.02 seconds (0.01 work units)

    Nodes    |    Current Node    |     Objective Bo